# Can we predict `passed_quiz`?
Follows `protocol.yaml`: group holdout on `student_key` (rows in `holdout.csv`), 5-fold GroupKFold CV on training rows, selection by mean CV ROC AUC, headline = chosen model refit on all training rows and scored once on the holdout rows.

Features: `student_key` is excluded (entity key). `enrol_year`, `school_code`, `prior_score` are known before the session; `session_minutes` and `hints_used` are recorded during the session, before the closing quiz result. Prediction is assumed to be made at the start of the closing quiz, so all five are kept.

In [ ]:
import json, numpy as np, pandas as pd
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
df=pd.read_csv('data.csv')
hold_idx=np.sort(pd.read_csv('holdout.csv')['row'].values)
train_idx=np.setdiff1d(np.arange(len(df)),hold_idx)
FEATURES=['enrol_year','school_code','prior_score','session_minutes','hints_used']
X,y,g=df[FEATURES],df['passed_quiz'],df['student_key']
Xtr,ytr,gtr=X.iloc[train_idx],y.iloc[train_idx],g.iloc[train_idx]
print(len(train_idx),len(hold_idx),ytr.mean())
print('student overlap train/holdout:',len(set(gtr)&set(g.iloc[hold_idx])))

## Candidates (all named before any scoring)

In [ ]:
candidates={
 'baseline_prior':lambda: DummyClassifier(strategy='prior'),
 'logreg_C1':lambda: make_pipeline(StandardScaler(),LogisticRegression(C=1.0,max_iter=1000)),
 'logreg_C0.1':lambda: make_pipeline(StandardScaler(),LogisticRegression(C=0.1,max_iter=1000)),
 'rf_depth6':lambda: RandomForestClassifier(n_estimators=300,max_depth=6,min_samples_leaf=20,random_state=0),
 'hgb_small':lambda: HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200,random_state=0),
}
print(list(candidates))

## Cross-validation on training rows (scaling lives inside each pipeline, so it is fit per fold)

In [ ]:
folds=list(GroupKFold(n_splits=5).split(Xtr,ytr,gtr))
cv_scores={}
for name,mk in candidates.items():
    cv_scores[name]=[float(roc_auc_score(ytr.iloc[va],mk().fit(Xtr.iloc[tr],ytr.iloc[tr]).predict_proba(Xtr.iloc[va])[:,1])) for tr,va in folds]
print(pd.DataFrame({k:{'mean':np.mean(v),'std':np.std(v)} for k,v in cv_scores.items()}).T)
chosen=max(cv_scores,key=lambda k:np.mean(cv_scores[k]))
print('chosen:',chosen)

## Refit on all training rows; score once on holdout

In [ ]:
final=candidates[chosen]().fit(Xtr,ytr)
auc=float(roc_auc_score(y.iloc[hold_idx],final.predict_proba(X.iloc[hold_idx])[:,1]))
print('holdout roc_auc of',chosen,':',auc)
evidence={
 'features':FEATURES,
 'candidates':list(candidates),
 'cv_folds':[[[int(train_idx[j]) for j in tr],[int(train_idx[j]) for j in va]] for tr,va in folds],
 'cv_scores':cv_scores,
 'chosen':chosen,
 'holdout_rows':[int(i) for i in hold_idx],
 'holdout_scores':{chosen:{'roc_auc':auc}},
}
json.dump(evidence,open('evidence.json','w'))